# Redrob Ranker - Sandbox Demo

Runnable demo of the **IndiaRun** hybrid candidate ranker
(repo: https://github.com/vinamravinay12/indiarun_datachallenge).

**Runtime -> Run all.** It loads a <=100-candidate sample, runs the full pipeline -
structured JD scoring + `bge-small` semantic fit + behavioral/logistics multipliers +
honeypot exclusion - and produces a ranked CSV, on CPU in well under 5 minutes.

The production ranker (`rank.py`) additionally gates to the strongest tier and refines with
verified skill-assessment over the full 100K pool; this demo shows the core hybrid mechanics
on a small sample (spec Section 10.5).

In [ ]:
# 1) setup: install the embedding lib, clone the repo, import the ranker core
!pip -q install sentence-transformers
!git clone -q https://github.com/vinamravinay12/indiarun_datachallenge.git
import sys; sys.path.insert(0, 'indiarun_datachallenge')
import json, numpy as np, pandas as pd
import ranker_core as rc
from sentence_transformers import SentenceTransformer
print('ready - ranker_core loaded')

### Load candidates (<=100)
Uses the bundled public 50-candidate sample by default. To rank your own, uncomment the upload
block and pick a `.json` (array) or `.jsonl` file with up to 100 candidate records.

In [ ]:
cands = json.load(open('indiarun_datachallenge/sandbox/sample_candidates.json'))

# --- optional: upload your own <=100-candidate file instead ---
# from google.colab import files
# up = files.upload()
# raw = next(iter(up.values())).decode('utf-8').strip()
# cands = json.loads(raw) if raw[:1]=='[' else [json.loads(l) for l in raw.splitlines() if l.strip()]

cands = cands[:100]
print(len(cands), 'candidates loaded')

In [ ]:
# 2) embed: JD query + each candidate's career prose (bge-small, CPU is fine for <=100)
model = SentenceTransformer(rc.EMB_MODEL); model.max_seq_length = rc.MAX_SEQ_LENGTH
jd_emb   = model.encode(rc.JD_QUERY, normalize_embeddings=True)
cand_ids = [c['candidate_id'] for c in cands]
cand_emb = model.encode([rc.work_text(c) for c in cands],
                        normalize_embeddings=True, show_progress_bar=True)
print('encoded', cand_emb.shape)

In [ ]:
# 3) features + honeypot exclusion + hybrid rank
df, hp = rc.build_feature_frame(cands)
feat = df.set_index('candidate_id')
print(f'{len(df)} candidates | honeypots flagged & excluded: {int(df.honeypot.sum())}')

top = rc.rank(feat, cand_emb, cand_ids, jd_emb)

if len(top) < 5:
    # Small samples are sparse in the top tier, so the production tier-5 gate may leave few rows.
    # Fall back to the core hybrid score over all non-honeypots so the demo always produces a ranking.
    sem = pd.Series(dict(zip(cand_ids, cand_emb @ jd_emb)))
    R = feat[~feat.honeypot].copy()
    R['sem_sim'] = sem
    R['struct_norm'] = (R.struct_score / max(R.struct_score.max(), 1e-9)).clip(0, 1)
    rng = max(R.sem_sim.max() - R.sem_sim.min(), 1e-9)
    R['sem_norm'] = ((R.sem_sim - R.sem_sim.min()) / rng).clip(0, 1)
    R['fit'] = 0.75*R.struct_norm + 0.25*R.sem_norm
    R['logistics'] = R.apply(rc.logistics_multiplier, axis=1)
    R['final'] = (R.fit * R.behavioral * R.logistics).round(4)
    top = R.sort_values('final', ascending=False).head(min(100, len(R)))

show = top.reset_index()[['candidate_id','current_title','current_industry','yoe',
                          'struct_score','behavioral','logistics','final']]
show.insert(0, 'rank', range(1, len(show)+1))
show.to_csv('sandbox_ranking.csv', index=False)
print('wrote sandbox_ranking.csv with', len(show), 'ranked candidates')
display(show.head(25))

# to download the CSV:
# from google.colab import files; files.download('sandbox_ranking.csv')